# Font Recognition — Training (Colab)

Runtime → Change runtime type → **GPU**.

Put ~30 `train-r*-p*.npz` files in the data folder (`fonts.csv`, `val-p*.npz`, `ref-p*.npz` stay there permanently) and run all cells. The batch trains for up to `max_epochs` epochs with early stopping. When it says the batch is finished, replace the train files with the next ~30 and run **Train on current batch** again.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

## 2. Imports

In [ ]:
import os, csv, glob, json, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

## 3. Settings

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    data = '/content/drive/MyDrive/fontdataset',          # .npz shards + fonts.csv
    out  = '/content/drive/MyDrive/FontDetection/runs/font_arcface', # checkpoints + progress.json
    total_files = 768,    # train shards in the full dataset (for progress display)
    max_epochs = 10,      # max epochs per batch of shards (cosine LR spans this)
    patience = 5,         # early stop: epochs without val improvement
    lr_patience = 2,      # cut LR after this many epochs without improvement (before early stop)
    lr_factor = 0.3,      # multiply LR by this on plateau
    min_delta = 0.002,    # smallest val top1 gain that counts as improvement
    warmup = 0.05,        # LR warm-up fraction (first batch)
    warmup_later = 0.02,  # LR warm-up fraction (later batches)
    lr_later = 1e-3,      # starting LR for later batches (lower = less forgetting)
    min_lr_frac = 0.01,   # LR decays to lr * this by the end of max_epochs
    batch = 512,
    lr = 2e-3,
    wd = 5e-4,
    emb_dim = 256,
    width = 64,
    margin = 0.25,
    scale = 30.0,
    workers = 2,
    buffer_files = 16,
    eval_files = 8,
    eval_ref_files = 0,
    log_every = 5,
    ckpt_every = 50,
    seed = 0,
    predict = None,
)

os.makedirs(args.out, exist_ok=True)
torch.manual_seed(args.seed)
print(len(glob.glob(f'{args.data}/train-r*-p*.npz')), 'train shards present,',
      len(glob.glob(f'{args.data}/val-p*.npz')), 'val,',
      len(glob.glob(f'{args.data}/ref-p*.npz')), 'ref')

## 4. Data

In [ ]:
class ShardStream(IterableDataset):
    """Streams images from .npz shards without loading the whole set.
    Each worker keeps `buffer_files` shards in RAM and samples randomly across
    them, so a batch mixes fonts from several style groups and recipes."""

    def __init__(self, files, buffer_files=6, shuffle=True, seed=0, augment=True):
        self.files, self.buffer_files = list(files), buffer_files
        self.shuffle, self.seed, self.augment = shuffle, seed, augment
        self.epoch = 0

    def set_epoch(self, e): self.epoch = e

    def _load(self, f):
        with np.load(f) as d:
            return d["x"], d["font"].astype(np.int64), d["char"].astype(np.int64)

    def _emit(self, X, F, C, rng):
        idx = rng.permutation(len(F)) if self.shuffle else np.arange(len(F))
        for i in idx:
            x = X[i]
            if self.augment and rng.random() < 0.5:      # small extra jitter
                dy, dx = rng.integers(-2, 3, 2)
                x = np.roll(np.roll(x, dy, 0), dx, 1)
            yield torch.from_numpy(np.ascontiguousarray(x)), F[i], C[i]

    def __iter__(self):
        wi = get_worker_info()
        wid, nw = (wi.id, wi.num_workers) if wi else (0, 1)
        rng = np.random.default_rng(self.seed + 1000 * self.epoch + wid)
        files = list(self.files)
        if self.shuffle:
            random.Random(self.seed + self.epoch).shuffle(files)
        files = files[wid::nw]
        for i in range(0, len(files), self.buffer_files):
            parts = [self._load(f) for f in files[i:i + self.buffer_files]]
            X = np.concatenate([p[0] for p in parts])
            F = np.concatenate([p[1] for p in parts])
            C = np.concatenate([p[2] for p in parts])
            del parts
            yield from self._emit(X, F, C, rng)

def count_images(files):
    n = 0
    for f in files:
        with np.load(f) as d: n += d["font"].shape[0]
    return n

def to_input(x_uint8, device):
    # (B,64,64) uint8 -> (B,1,64,64) float in [-1,1]
    return (x_uint8.to(device, non_blocking=True).float().unsqueeze(1) / 127.5 - 1.0)

## 5. Model

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
                   if stride != 1 or cin != cout else nn.Identity())

    def forward(self, x):
        y = Fn.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return Fn.relu(y + self.sc(x))

class FontEncoder(nn.Module):
    def __init__(self, emb_dim=256, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(1, w // 2, 3, 1, 1, bias=False), nn.BatchNorm2d(w // 2),
                                  nn.ReLU(),                                  # 64x64 stroke edges
                                  nn.Conv2d(w // 2, w, 3, 2, 1, bias=False), nn.BatchNorm2d(w),
                                  nn.ReLU())
        self.layers = nn.Sequential(
            Block(w, w, 1),         Block(w, w, 1),          # 32x32
            Block(w, 2 * w, 2),     Block(2 * w, 2 * w, 1),  # 16x16
            Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1),  # 8x8
            Block(4 * w, 8 * w, 2), Block(8 * w, 8 * w, 1),  # 4x4
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.1), nn.Linear(8 * w * 2, emb_dim),
                                  nn.BatchNorm1d(emb_dim))

    def forward(self, x):
        h = self.layers(self.stem(x))
        h = torch.cat([Fn.adaptive_avg_pool2d(h, 1), Fn.adaptive_max_pool2d(h, 1)], 1)
        return Fn.normalize(self.head(h), dim=1)

class ArcFace(nn.Module):
    def __init__(self, emb_dim, n_classes, s=30.0, m=0.25):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_classes, emb_dim) * 0.01)
        self.s, self.m = s, m

    def logits(self, emb):
        return emb @ Fn.normalize(self.W, dim=1).t()

    def forward(self, emb, y):
        cos = self.logits(emb).float().clamp(-1 + 1e-7, 1 - 1e-7)
        theta = torch.acos(cos.gather(1, y[:, None]))
        target = torch.cos(theta + self.m)
        cos = cos.scatter(1, y[:, None], target)
        return Fn.cross_entropy(cos * self.s, y, label_smoothing=0.05)

## 6. Evaluation

In [ ]:
@torch.no_grad()
def embed_files(model, files, device, batch=1024):
    model.eval()
    E, F, C = [], [], []
    loader = DataLoader(ShardStream(files, buffer_files=1, shuffle=False, augment=False),
                        batch_size=batch, num_workers=0)
    for x, f, c in loader:
        E.append(model(to_input(x, device)).cpu()); F.append(f); C.append(c)
    return torch.cat(E), torch.cat(F), torch.cat(C)

def build_prototypes(E, F, n_classes):
    P = torch.zeros(n_classes, E.shape[1])
    P.index_add_(0, F, E)
    cnt = torch.bincount(F, minlength=n_classes).clamp(min=1)[:, None]
    return Fn.normalize(P / cnt, dim=1), torch.bincount(F, minlength=n_classes) > 0

@torch.no_grad()
def evaluate(model, head, val_files, ref_files, n_classes, device):
    Ev, Fv, _ = embed_files(model, val_files, device)
    out = {}
    lg = head.logits(Ev.to(device)).cpu()
    top = lg.topk(5, 1).indices
    out["cls_top1"] = (top[:, 0] == Fv).float().mean().item()
    out["cls_top5"] = (top == Fv[:, None]).any(1).float().mean().item()
    if not ref_files: return out
    Er, Fr, _ = embed_files(model, ref_files, device)
    P, has = build_prototypes(Er, Fr, n_classes)
    sim = Ev @ P.t(); sim[:, ~has] = -2
    top = sim.topk(5, 1).indices
    out["proto_top1"] = (top[:, 0] == Fv).float().mean().item()
    out["proto_top5"] = (top == Fv[:, None]).any(1).float().mean().item()
    return out

## 7. Helpers

In [ ]:
def load_styles(data_dir):
    with open(os.path.join(data_dir, "fonts.csv"), newline="", encoding="utf-8") as fh:
        return [(r["family"], r["style"]) for r in csv.DictReader(fh)]

def pick_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def save_ckpt(path, **state):
    tmp = path + ".tmp"
    torch.save(state, tmp); os.replace(tmp, path)

## 8. Training loop

In [ ]:
# Each run trains the train shards currently in `args.data` for up to
# `max_epochs` epochs, stopping early when the val accuracy on this batch's
# fonts stops improving. The best weights of the batch carry over to the next.
import re

def shard_p(name): return re.search(r"-p(\d+)\.npz$", name).group(1)

def lr_at_step(base_lr, s, total, warmup, min_frac):
    w = max(1, int(total * warmup))
    if s < w: return base_lr * max(s / w, 0.02)
    p = min(1.0, (s - w) / max(total - w, 1))
    return base_lr * (min_frac + (1 - min_frac) * 0.5 * (1 + math.cos(math.pi * p)))

def load_prog(path):
    prog = dict(batches=[], current=None)
    if os.path.exists(path):
        with open(path) as fh: prog.update(json.load(fh))
    return prog

def save_prog(path, prog):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh: json.dump(prog, fh, indent=1)
    os.replace(tmp, path)

def expand_arcface(ck_head_state, old_n, new_n, emb_dim):
    """Expand ArcFace weight matrix to accommodate new font classes."""
    old_W = ck_head_state["W"]
    new_W = torch.randn(new_n, emb_dim) * 0.01
    new_W[:old_n] = old_W
    ck_head_state["W"] = new_W
    return ck_head_state

def status(args):
    prog = load_prog(os.path.join(args.out, "progress.json"))
    done = len({f for b in prog["batches"] for f in b["files"]})
    print(f"{len(prog['batches'])} batches finished, {done}/{args.total_files} shards trained")
    for i, b in enumerate(prog["batches"], 1):
        print(f"  batch {i}: {len(b['files'])} shards, {b['epochs']} epochs, "
              f"batch-val top1 {b['best']:.4f}, all-fonts top1 {b['all_top1']:.4f}")
    c = prog["current"]
    if c and not c["done"]:
        print(f"current batch in progress: epoch {c['epoch']}/{args.max_epochs}, best {c['best']:.4f}, "
              f"{c['bad']}/{args.patience} epochs without improvement")

def train_batch(args):
    device = pick_device()
    n_classes = len(load_styles(args.data))
    names = sorted(os.path.basename(f) for f in glob.glob(os.path.join(args.data, "train-r*-p*.npz")))
    if not names:
        print("no train shards found in", args.data); return
    prog_path = os.path.join(args.out, "progress.json")
    last, batch_best = os.path.join(args.out, "last.pt"), os.path.join(args.out, "batch_best.pt")
    prog = load_prog(prog_path)
    cur = prog["current"]
    if cur and cur["files"] == names and cur["done"]:
        print("re-training same batch to continue improving from previous best weights")
        cur = dict(files=names, epoch=0, best=cur["best"], bad=0, done=False, history=[],
                   lr_drops=0)
        prog["current"] = cur; save_prog(prog_path, prog)
    seen_before = {f for b in prog["batches"] for f in b["files"]}
    if any(f in seen_before for f in names):
        print(f"note: {sum(f in seen_before for f in names)} of these shards were trained in an earlier batch")
    if not cur or cur["files"] != names:
        if cur and not cur["done"]:
            print("note: the previous batch was not finished; starting this batch from its latest weights")
        cur = dict(files=names, epoch=0, best=-1.0, bad=0, done=False, history=[],
                   lr_drops=0)
        prog["current"] = cur; save_prog(prog_path, prog)
    cur.setdefault("lr_drops", 0)
    first = len(prog["batches"]) == 0
    base_lr = args.lr if first else args.lr_later
    warmup = args.warmup if first else args.warmup_later

    files = [os.path.join(args.data, f) for f in names]
    val_all = sorted(glob.glob(os.path.join(args.data, "val-p*.npz")))
    val_batch = [v for v in (os.path.join(args.data, f"val-p{q}.npz") for q in sorted({shard_p(f) for f in names}))
                 if os.path.exists(v)] or val_all
    val_all = val_all[::max(1, len(val_all) // args.eval_files)][:args.eval_files]
    n_imgs = count_images(files)
    steps = n_imgs // args.batch
    total = steps * args.max_epochs
    print(f"device {device} | batch {len(prog['batches']) + 1}: {len(files)} shards, {n_imgs:,} imgs, "
          f"{steps:,} steps/epoch | up to {args.max_epochs} epochs, early stop after {args.patience} "
          f"| LR cut after {args.lr_patience} stale epochs"
          f"| early-stop val: {len(val_batch)} val shards for this batch's fonts")

    model = FontEncoder(args.emb_dim, args.width).to(device)
    head = ArcFace(args.emb_dim, n_classes, s=args.scale, m=args.margin).to(device)
    if device.type == "cuda": model = model.to(memory_format=torch.channels_last)
    params = list(model.parameters()) + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=base_lr, weight_decay=args.wd)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cfg = dict(emb_dim=args.emb_dim, width=args.width, n_classes=n_classes)

    skip = 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=True)
        old_cfg = ck["cfg"]
        if old_cfg["emb_dim"] != cfg["emb_dim"] or old_cfg["width"] != cfg["width"]:
            raise RuntimeError(f"last.pt was trained with emb_dim={old_cfg['emb_dim']}, width={old_cfg['width']}, "
                               f"but settings ask for emb_dim={cfg['emb_dim']}, width={cfg['width']}. "
                               "Use the same width/emb_dim, or a different `out` folder.")
        model.load_state_dict(ck["model"])
        old_n = old_cfg["n_classes"]
        if old_n != n_classes:
            ck["head"] = expand_arcface(ck["head"], old_n, n_classes, cfg["emb_dim"])
            print(f"expanded ArcFace head: {old_n} → {n_classes} classes ({n_classes - old_n} new)")
        head.load_state_dict(ck["head"])
        if ck.get("files") == names and "opt" in ck:
            if old_n == n_classes:
                opt.load_state_dict(ck["opt"])
            else:
                print("optimizer state reset (class count changed)")
            if ck.get("epoch") == cur["epoch"]: skip = ck.get("step_in_epoch", 0)
            print(f"resuming this batch at epoch {cur['epoch'] + 1}, step {skip}/{steps}")
        else:
            print("starting from the weights of the previous batch")

    def ck_state(epoch, step_in_epoch):
        return dict(model=model.state_dict(), head=head.state_dict(), opt=opt.state_dict(), cfg=cfg,
                    files=names, epoch=epoch, step_in_epoch=step_in_epoch)

    lr_scale = args.lr_factor ** cur["lr_drops"]
    effective_base = base_lr * lr_scale
    gstep, t_batch = cur["epoch"] * steps + skip, time.time()
    for ep in range(cur["epoch"], args.max_epochs):
        if cur["bad"] >= args.patience: break
        ds = ShardStream(files, buffer_files=args.buffer_files, seed=args.seed + len(prog["batches"]))
        ds.set_epoch(ep)
        loader = DataLoader(ds, batch_size=args.batch, num_workers=args.workers, drop_last=True,
                            pin_memory=device.type == "cuda",
                            prefetch_factor=4 if args.workers else None)
        model.train(); head.train()
        t0, t_ep, run_loss, run_n, lr = time.time(), time.time(), 0.0, 0, effective_base
        for i, (x, y, _) in enumerate(loader):
            if i >= steps: break
            if i < skip:
                t0 = time.time(); continue
            cosine_lr = lr_at_step(base_lr, gstep, total, warmup, args.min_lr_frac)
            lr = cosine_lr * lr_scale
            for g in opt.param_groups: g["lr"] = lr
            x = to_input(x, device); y = y.to(device, non_blocking=True)
            if device.type == "cuda": x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                emb = model(x)
            loss = head(emb.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, 5.0)
            scaler.step(opt); scaler.update()
            gstep += 1; run_loss += loss.item(); run_n += 1
            if (i + 1) % args.log_every == 0:
                ips = run_n * args.batch / (time.time() - t0)
                print(f"epoch {ep + 1} step {i + 1}/{steps} loss {run_loss / run_n:.3f} lr {lr:.2e} "
                      f"{ips:.0f} img/s", flush=True)
                t0, run_loss, run_n = time.time(), 0.0, 0
            if args.ckpt_every and (i + 1) % args.ckpt_every == 0:
                save_ckpt(last, **ck_state(ep, i + 1))
        skip = 0

        m = evaluate(model, head, val_batch, None, n_classes, device)
        cur["epoch"] = ep + 1
        if m["cls_top1"] > cur["best"] + args.min_delta:
            cur["best"], cur["bad"] = m["cls_top1"], 0
            save_ckpt(batch_best, model=model.state_dict(), head=head.state_dict(), cfg=cfg)
            note = "new best"
        else:
            cur["bad"] += 1
            if cur["bad"] >= args.lr_patience and cur["bad"] < args.patience:
                cur["lr_drops"] = cur.get("lr_drops", 0) + 1
                lr_scale = args.lr_factor ** cur["lr_drops"]
                effective_base = base_lr * lr_scale
                note = (f"no improvement ({cur['bad']}/{args.patience}), "
                        f"LR cut #{cur['lr_drops']} -> {effective_base:.2e}")
            else:
                note = f"no improvement ({cur['bad']}/{args.patience})"
        cur["history"].append(dict(epoch=ep + 1, lr=lr, **m))
        save_ckpt(last, **ck_state(ep + 1, 0))
        save_prog(prog_path, prog)
        print(f"== epoch {ep + 1}/{args.max_epochs} done in {(time.time() - t_ep) / 60:.1f}m | "
              + "  ".join(f"{k} {v:.4f}" for k, v in m.items()) + f" | {note}", flush=True)
        if cur["bad"] >= args.patience:
            print(f"early stopping: no improvement for {args.patience} epochs")
            break

    bk = torch.load(batch_best, map_location=device, weights_only=True)
    model.load_state_dict(bk["model"]); head.load_state_dict(bk["head"])
    m_all = evaluate(model, head, val_all, None, n_classes, device)
    save_ckpt(last, model=model.state_dict(), head=head.state_dict(), cfg=cfg, files=names, epoch=-1)
    save_ckpt(os.path.join(args.out, "best.pt"), model=model.state_dict(), head=head.state_dict(), cfg=cfg)
    cur["done"] = True
    prog["batches"].append(dict(files=names, epochs=cur["epoch"], best=cur["best"],
                                all_top1=m_all["cls_top1"], all_top5=m_all["cls_top5"]))
    save_prog(prog_path, prog)
    done = len({f for b in prog["batches"] for f in b["files"]})
    print(f"batch finished in {(time.time() - t_batch) / 60:.0f}m: best batch-val top1 {cur['best']:.4f}, "
          f"all-fonts top1 {m_all['cls_top1']:.4f} top5 {m_all['cls_top5']:.4f}")
    print(f"{done}/{args.total_files} shards trained. Replace the train shards with the next batch "
          "and run this cell again.")

## 9. Index + predict

In [ ]:
def load_model(path, device):
    ck = torch.load(path, map_location=device, weights_only=True)
    cfg = ck["cfg"]
    model = FontEncoder(cfg["emb_dim"], cfg["width"]).to(device)
    model.load_state_dict(ck["model"]); model.eval()
    return model, cfg

def build_index(args):
    device = pick_device()
    ckpt = os.path.join(args.out, "best.pt")
    if not os.path.exists(ckpt): ckpt = os.path.join(args.out, "last.pt")
    model, cfg = load_model(ckpt, device)
    ref_files = sorted(glob.glob(os.path.join(args.data, "ref-p*.npz")))
    print(f"building font index from {len(ref_files)} ref files with {os.path.basename(ckpt)}...")
    E, F, C = embed_files(model, ref_files, device)
    P, has = build_prototypes(E, F, cfg["n_classes"])
    styles = load_styles(args.data)
    path = os.path.join(args.out, "font_index.npz")
    np.savez(path, prototypes=P.numpy().astype(np.float32), valid=has.numpy(),
             family=np.array([s[0] for s in styles]), style=np.array([s[1] for s in styles]))
    print(f"saved {path} ({int(has.sum())} fonts)")

def predict(args):
    from PIL import Image, ImageOps
    device = pick_device()
    ckpt = os.path.join(args.out, "best.pt")
    if not os.path.exists(ckpt): ckpt = os.path.join(args.out, "last.pt")
    model, _ = load_model(ckpt, device)
    idx = np.load(os.path.join(args.out, "font_index.npz"))
    img = Image.open(args.predict).convert("L")
    img = ImageOps.pad(img, (64, 64), color=int(np.median(np.asarray(img))))
    x = torch.from_numpy(np.asarray(img, np.uint8))[None]
    with torch.no_grad():
        e = model(to_input(x, device)).cpu().numpy()[0]
    sim = idx["prototypes"] @ e
    sim[~idx["valid"]] = -2
    for r, i in enumerate(np.argsort(-sim)[:5], 1):
        print(f"{r}. {idx['family'][i]} {idx['style'][i]}  ({sim[i]:.3f})")

## 10. Status

In [ ]:
status(args)

## 11. Train on current batch

In [ ]:
train_batch(args)

## 12. Build font index

In [ ]:
build_index(args)

## 13. Predict

In [ ]:
from google.colab import files
for name in files.upload():
    args.predict = name
    predict(args)